In [1]:
from pathlib import Path
import re
import numpy as np
import pandas as pd

# Settings
pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 140)

# Paths
BASE = Path("../../..")
OUT_DIR = BASE / "data" / "output"
OUT_DIR.mkdir(parents=True, exist_ok=True)

In [2]:
def _clean_colnames(cols):
    """Standardize column names: lower-case, strip, replace spaces/punct with _."""
    out = []
    for c in cols:
        c2 = re.sub(r"[^0-9a-zA-Z]+", "_", str(c).strip()).strip("_").lower()
        out.append(c2)
    return out

In [9]:
# Read penetration data
def read_penetration(path: str) -> pd.DataFrame:
    df = pd.read_csv(
        path,
        skiprows=1,
        names=[
            "state", "county", "fips_state", "fips_cnty", "fips",
            "ssa_state", "ssa_cnty", "ssa", "eligibles", "enrolled", "penetration"
        ],
        dtype={
            "state": "string", "county": "string",
            "fips_state": "Int64", "fips_cnty": "Int64", "fips": "float64",
            "ssa_state": "Int64", "ssa_cnty": "Int64", "ssa": "float64",
            "eligibles": "string", "enrolled": "string", "penetration": "string"
        },
        na_values=["", "NA", "*", "-", "--", "UK"],
    )

    for c in ["eligibles", "enrolled", "penetration"]:
        df[c] = (
            df[c]
            .astype("string")
            .str.replace(",", "", regex=False)
            .str.extract(r"(-?\d+(?:\.\d+)?)")[0]
            .astype("float64")
        )

    return df

In [6]:
# Build penetration data for one year
def build_year_penetration(year: int) -> pd.DataFrame:
    print(f"  Reading penetration data for {year}...")

    MONTHS = [f"{m:02d}" for m in range(1, 13)]
    pen_dir = BASE / "data" / "input" / f"penetration_{year}"

    months_data = []
    for m in MONTHS:
        path = pen_dir / f"State_County_Penetration_MA_{year}_{m}.csv"
        if not path.exists():
            print(f"    WARNING: Missing penetration file: {path.name}")
            continue
        df = read_penetration(str(path))
        df["month"] = int(m)
        df["year"] = year
        months_data.append(df)

    if not months_data:
        print(f"  ERROR: No penetration files found for {year}")
        return pd.DataFrame()

    ma_penetration = pd.concat(months_data, ignore_index=True)
    ma_penetration = ma_penetration.sort_values(["state", "county", "month"])

    # Fill missing FIPS within state/county
    ma_penetration["fips"] = (
        ma_penetration
        .groupby(["state", "county"], dropna=False)["fips"]
        .transform(lambda s: s.ffill().bfill())
    )

    # Collapse to one row per (fips, state, county, year)
    final_penetration = (
        ma_penetration
        .sort_values(["fips", "state", "county", "year", "month"])
        .groupby(["fips", "state", "county", "year"], dropna=False)
        .agg(
            n_elig=("eligibles", lambda s: s.notna().sum()),
            avg_eligibles=("eligibles", lambda s: s.mean(skipna=True)),
            sd_eligibles=("eligibles", lambda s: s.std(skipna=True)),
            min_eligibles=("eligibles", "min"),
            max_eligibles=("eligibles", "max"),
            first_eligibles=("eligibles", lambda s: s.dropna().iloc[0] if s.notna().any() else pd.NA),
            last_eligibles=("eligibles", lambda s: s.dropna().iloc[-1] if s.notna().any() else pd.NA),

            n_enrol=("enrolled", lambda s: s.notna().sum()),
            avg_enrolled=("enrolled", lambda s: s.mean(skipna=True)),
            sd_enrolled=("enrolled", lambda s: s.std(skipna=True)),
            min_enrolled=("enrolled", "min"),
            max_enrolled=("enrolled", "max"),
            first_enrolled=("enrolled", lambda s: s.dropna().iloc[0] if s.notna().any() else pd.NA),
            last_enrolled=("enrolled", lambda s: s.dropna().iloc[-1] if s.notna().any() else pd.NA),

            ssa=("ssa", "last"),
        )
        .reset_index()
    )

    print(f"  Penetration data: {len(final_penetration):,} counties")
    return final_penetration

In [7]:
# Merge penetration data into plan data and calculate market share and HHI
def build_year_with_penetration(year: int) -> pd.DataFrame:
    # Load already-built plan + service area data
    plan_path = OUT_DIR / f"ma_data_{year}.csv"
    if not plan_path.exists():
        print(f"ERROR: Plan data not found for {year}. Run build_year_ma() first.")
        return pd.DataFrame()

    plan_df = pd.read_csv(plan_path, low_memory=False)
    print(f"\n{'='*60}")
    print(f"Merging penetration data for {year} ({len(plan_df):,} plan rows)")
    print(f"{'='*60}")

    # Build penetration data
    pen_df = build_year_penetration(year)
    if pen_df.empty:
        return pd.DataFrame()

    # Merge on fips + year
    merged = plan_df.merge(
        pen_df[["fips", "year", "avg_enrolled", "avg_eligibles", "ssa"]],
        on=["fips", "year"],
        how="left",
        suffixes=("", "_pen")
    )

    print(f"  After penetration merge: {len(merged):,} rows")
    print(f"  Rows missing avg_enrolled: {merged['avg_enrolled'].isna().sum():,}")

    # --- Market share: plan enrollment / total county MA enrollment ---
    merged["enrollment"] = pd.to_numeric(merged["enrollment"], errors="coerce")
    merged["market_share"] = np.where(
        merged["avg_enrolled"] > 0,
        merged["enrollment"] / merged["avg_enrolled"],
        np.nan
    )
    print(f"  Market share computed using 'enrollment' / avg_enrolled")

    # --- HHI: sum of squared market shares by county/year ---
    merged["market_share_sq"] = merged["market_share"] ** 2
    hhi = (
        merged
        .groupby(["fips", "year"], dropna=False)["market_share_sq"]
        .sum()
        .reset_index()
        .rename(columns={"market_share_sq": "hhi"})
    )
    merged = merged.merge(hhi, on=["fips", "year"], how="left")
    print(f"  HHI computed (uses avg_enrolled as denominator)")

    # Save updated file
    out_path = OUT_DIR / f"ma_data_{year}_with_penetration.csv"
    merged.to_csv(out_path, index=False)
    print(f"  Saved to: {out_path}")

    return merged

In [8]:
def build_all_years_with_penetration(years: list = [2010, 2011, 2012, 2013, 2014, 2015]):
    results = {}

    for year in years:
        try:
            df = build_year_with_penetration(year)
            results[year] = df
        except Exception as e:
            print(f"\n✗ ERROR processing {year}: {e}")
            results[year] = pd.DataFrame()

    print(f"\n{'='*60}")
    print("Summary:")
    print(f"{'='*60}")
    for year, df in results.items():
        status = "✓" if not df.empty else "✗"
        print(f"{status} {year}: {len(df):,} rows")

    return results

# Run it
results_with_pen = build_all_years_with_penetration()


Merging penetration data for 2010 (112,856 plan rows)
  Reading penetration data for 2010...

✗ ERROR processing 2010: Unable to parse string "UK" at position 3276

Merging penetration data for 2011 (70,184 plan rows)
  Reading penetration data for 2011...
  Penetration data: 3,228 counties
  After penetration merge: 70,184 rows
  Rows missing avg_enrolled: 429
  Market share computed using 'enrollment' / avg_enrolled
  HHI computed (uses avg_enrolled as denominator)
  Saved to: ../../../data/output/ma_data_2011_with_penetration.csv

Merging penetration data for 2012 (69,770 plan rows)
  Reading penetration data for 2012...
  Penetration data: 3,225 counties
  After penetration merge: 69,770 rows
  Rows missing avg_enrolled: 381
  Market share computed using 'enrollment' / avg_enrolled
  HHI computed (uses avg_enrolled as denominator)
  Saved to: ../../../data/output/ma_data_2012_with_penetration.csv

Merging penetration data for 2013 (70,930 plan rows)
  Reading penetration data for 

In [10]:
results_with_pen[2010] = build_year_with_penetration(2010)


Merging penetration data for 2010 (112,856 plan rows)
  Reading penetration data for 2010...
  Penetration data: 3,280 counties
  After penetration merge: 112,856 rows
  Rows missing avg_enrolled: 534
  Market share computed using 'enrollment' / avg_enrolled
  HHI computed (uses avg_enrolled as denominator)
  Saved to: ../../../data/output/ma_data_2010_with_penetration.csv
